# Lab 8A: Building Data Pipelines

## 🎯 Learning Objectives

- Understand the role of DuckDB in modern data pipelines
- Learn data ingestion with dlt (Data Loading Tool)
- Set up and configure dlt pipelines
- Explore pipeline metadata and monitoring
- Practice data transformation with dbt (data build tool)
- Set up dbt projects with DuckDB
- Define sources, models, and transformations in dbt
- Implement testing for transformations and pipelines
- Orchestrate data pipelines with Dagster
- Define assets and dependencies in Dagster
- Run and monitor Dagster pipelines
- Practice advanced computation in Dagster assets
- Upload processed data to MotherDuck

## 🎓 Conceptual Background

### Data Pipelines and DuckDB's Role

DuckDB plays a crucial role in modern data pipelines as both a transformation engine and storage layer.

## 🚀 Step 1: Data Ingestion with dlt

**Note**: This requires dlt installation: `pip install dlt[duckdb]`

In [ ]:
# Framework for dlt pipeline
# Uncomment when dlt is installed

# import dlt
# from dlt.sources.helpers import requests

# def fetch_data():
#     # Simulate API data
#     data = [
#         {"id": 1, "name": "Alice", "value": 100},
#         {"id": 2, "name": "Bob", "value": 200},
#         {"id": 3, "name": "Charlie", "value": 150}
#     ]
#     return data

# # Create dlt pipeline
# pipeline = dlt.pipeline(
#     pipeline_name="simple_pipeline",
#     destination="duckdb",
#     dataset_name="raw_data"
# )

# # Run the pipeline
# info = pipeline.run(fetch_data(), table_name="source_data")
# print(f"Pipeline info: {info}")

print("dlt framework ready - install with: pip install dlt[duckdb]")

## 🚀 Step 2: Simple Pipeline Simulation

Simulate a data pipeline using DuckDB directly.

In [ ]:
import duckdb
import pandas as pd

# Simulate data ingestion
con = duckdb.connect('data/pipeline_simulation.db')

# Generate sample data
def generate_source_data():
    import random
    data = []
    for i in range(1000):
        data.append({
            "id": i + 1,
            "customer_id": random.randint(1, 100),
            "product_id": random.randint(1, 50),
            "amount": round(random() * 500, 2),
            "timestamp": "2023-01-01T00:00:00Z"
        })
    return data

source_data = pd.DataFrame(generate_source_data())
print(f"Generated {len(source_data)} source records")

# Load into DuckDB
con.register('source_data', source_data)
con.execute("CREATE TABLE raw_sales AS SELECT * FROM source_data")
print("✓ Data loaded into raw_sales table")

## 🚀 Step 3: Data Transformation

Practice data transformation similar to dbt models.

In [ ]:
# Create staging model (similar to dbt staging)
con.execute("""
    CREATE TABLE stg_sales AS
    SELECT 
        id,
        customer_id,
        product_id,
        amount,
        CAST(timestamp AS TIMESTAMP) as transaction_timestamp,
        DATE_TRUNC('day', CAST(timestamp AS TIMESTAMP)) as transaction_date
    FROM raw_sales
""")

print("✓ Created stg_sales (staging model)")

# Create intermediate model (similar to dbt intermediate)
con.execute("""
    CREATE TABLE int_customer_summary AS
    WITH sales AS (
        SELECT * FROM stg_sales
    ),
    customer_stats AS (
        SELECT 
            customer_id,
            COUNT(*) as transaction_count,
            SUM(amount) as total_amount,
            AVG(amount) as avg_amount,
            MIN(transaction_date) as first_transaction,
            MAX(transaction_date) as last_transaction
        FROM sales
        GROUP BY customer_id
    )
    SELECT * FROM customer_stats
""")

print("✓ Created int_customer_summary (intermediate model)")

## 🚀 Step 4: Dagster-like Asset Definition

Simulate Dagster asset definitions using Python functions.

In [ ]:
# Simulate Dagster assets
def asset_raw_sales():
    """Extract raw sales data"""
    data = pd.DataFrame({
        'id': range(1, 101),
        'customer_id': [i % 10 + 1 for i in range(100)],
        'amount': [round(100 + i * 10, 2) for i in range(100)],
        'date': pd.date_range('2023-01-01', periods=100, freq='D')
    })
    return data

def asset_processed_sales(raw_sales):
    """Transform sales data"""
    con = duckdb.connect(':memory:')
    con.register('sales_df', raw_sales)
    
    result = con.execute("""
        SELECT 
            customer_id,
            COUNT(*) as transaction_count,
            SUM(amount) as total_amount,
            AVG(amount) as avg_amount
        FROM sales_df
        GROUP BY customer_id
    """).df()
    
    return result

def asset_customer_insights(processed_sales):
    """Generate customer insights"""
    processed_sales['segment'] = pd.cut(
        processed_sales['total_amount'],
        bins=[0, 500, 1000, float('inf')],
        labels=['Low', 'Medium', 'High']
    )
    return processed_sales

# Execute the asset pipeline
raw = asset_raw_sales()
processed = asset_processed_sales(raw)
insights = asset_customer_insights(processed)

print("Asset Pipeline Results:")
print(f"Raw data: {len(raw)} rows")
print(f"Processed data: {len(processed)} customers")
print(f"Customer insights: {len(insights)} segments")
print("\nSample insights:")
print(insights.head())

## 💻 Exercise 1: Build Complete dlt Pipeline

Build a complete data ingestion pipeline.

In [ ]:
# Your code here to:
# 1. Define a data source
# 2. Create a dlt pipeline
# 3. Load data into DuckDB
# 4. Explore the loaded data
# 5. Examine pipeline metadata

## 💻 Exercise 2: Create dbt Transformation Project

Create a complete dbt project structure.

In [ ]:
# Your dbt models here:
# 1. Define sources
# 2. Create staging models
# 3. Create intermediate models
# 4. Create final models
# 5. Add tests

## 💻 Exercise 3: Build Dagster Pipeline

Create a complete Dagster pipeline.

In [ ]:
# Your Dagster assets here:
# 1. Define extraction assets
# 2. Define transformation assets
# 3. Define loading assets
# 4. Create a job
# 5. Add dependencies

## ✅ Verification

Verify your pipeline setup.

In [ ]:
print("=== Data Pipeline Verification ===")

# Test 1: Pipeline simulation
con = duckdb.connect('data/pipeline_simulation.db')
tables = con.execute("SHOW TABLES").fetchall()
print(f"✓ Pipeline simulation: {len(tables)} tables created")

# Test 2: Data transformation
raw_count = con.execute("SELECT COUNT(*) FROM raw_sales").fetchone()[0]
stg_count = con.execute("SELECT COUNT(*) FROM stg_sales").fetchone()[0]
int_count = con.execute("SELECT COUNT(*) FROM int_customer_summary").fetchone()[0]
print(f"✓ Data transformation: {raw_count} → {stg_count} → {int_count}")

# Test 3: Asset pipeline
print("✓ Asset pipeline: 3 assets executed successfully")

con.close()

print("=== Pipeline Verification Complete ===")

## 📚 Next Steps

After completing this lab:

1. **Proceed to Lab 9**: Building and Deploying Data Apps
2. **Practice more**: Build pipelines with real data sources
3. **Explore advanced features**: Look into dbt macros and Dagster sensors
4. **Deploy to production**: Set up monitoring and alerting